In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SLIM = ROOT / "data" / "processed" / "loans_slim.parquet"

loans = pd.read_parquet(SLIM)
print(loans.shape)

(2260668, 37)


In [2]:
SNAPSHOT = pd.Timestamp("2019-03-01")

loans["months_on_book"] = (
    (SNAPSHOT.year - loans["issue_d"].dt.year) * 12
    + (SNAPSHOT.month - loans["issue_d"].dt.month)
)
loans["matured"] = loans["months_on_book"] >= loans["term"]

loans["matured"].value_counts()

matured
False    1474753
True      785915
Name: count, dtype: int64

In [3]:
loans["issue_year"] = loans["issue_d"].dt.year

loans.pivot_table(
    index="issue_year", columns="term", values="matured", aggfunc="mean"
).round(2)

term,36,60
issue_year,,
2007,1.0,NaN
2008,1.0,NaN
2009,1.0,NaN
2010,1.0,1.00
2011,1.0,1.00
2012,1.0,1.00
2013,1.0,1.00
2014,1.0,0.18
2015,1.0,0.00


In [4]:
pd.crosstab(loans["loan_status"], loans["matured"], margins=True)

matured,False,True,All
loan_status,,,
Charged Off,150949,117610,268559
Current,872369,5948,878317
Default,39,1,40
Does not meet the credit policy. Status:Charged Off,0,761,761
Does not meet the credit policy. Status:Fully Paid,0,1988,1988
Fully Paid,418085,658666,1076751
In Grace Period,8226,210,8436
Late (16-30 days),4248,101,4349
Late (31-120 days),20837,630,21467


In [5]:
OUTCOME = {
    "Fully Paid": 0,
    "Does not meet the credit policy. Status:Fully Paid": 0,
    "Charged Off": 1,
    "Does not meet the credit policy. Status:Charged Off": 1,
    "Default": 1,
}

loans["bad"] = loans["loan_status"].map(OUTCOME).astype("Int64")
loans.loc[~loans["matured"], "bad"] = pd.NA
loans["old_policy"] = loans["loan_status"].str.startswith("Does not meet")

In [6]:
print(loans["bad"].value_counts(dropna=False))

judged = loans[loans["bad"].notna()]
print(f"Judged loans: {len(judged):,}")
print(f"Default rate: {judged['bad'].mean():.2%}")

bad
<NA>    1481642
0        660654
1        118372
Name: count, dtype: Int64
Judged loans: 779,026
Default rate: 15.19%


In [7]:
unresolved = loans["matured"] & loans["bad"].isna()
n_unres = unresolved.sum()
n_bad = judged["bad"].sum()
total = len(judged) + n_unres

print(f"Unresolved matured loans: {n_unres:,}")
print(f"If all were good: {n_bad / total:.2%}")
print(f"If all were bad:  {(n_bad + n_unres) / total:.2%}")

Unresolved matured loans: 6,889
If all were good: 15.06%
If all were bad:  15.94%


In [8]:
CLEAN = ROOT / "data" / "processed" / "loans_clean.parquet"
loans.to_parquet(CLEAN, index=False)
print(loans.shape)

(2260668, 42)
